In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import pandas as pd
from src.utils.pipeline import load_all_snapshots

df = load_all_snapshots()
print(df.shape)

cols = ["plate_x", "plate_z", "sz_top", "sz_bot", "zone", "stand"]
print(df[cols].dtypes)
print()
print(df[cols].isna().mean())
print()
df[cols].describe()

(710632, 119)
plate_x    object
plate_z    object
sz_top     object
sz_bot     object
zone       object
stand      object
dtype: object

plate_x    0.003753
plate_z    0.003753
sz_top     0.003753
sz_bot     0.003753
zone       0.003753
stand      0.000000
dtype: float64



,plate_x,plate_z,sz_top,sz_bot,zone,stand
count,707965.000000,707965.000000,707965.00,707965.00,707965,710632
unique,707965.000000,707965.000000,320947.00,313083.00,13,2
top,0.439332,3.164778,3.41,1.63,14,R
freq,1.000000,1.000000,8713.00,14667.00,133369,405035


In [2]:
by_batter = (
    df.groupby("batter")[["sz_top", "sz_bot"]]
    .agg(["mean", "std", "count"])
    .round(3)
)
by_batter = by_batter[by_batter[("sz_top", "count")] >= 20]
print(by_batter.head(10).to_string())
print()
print("sz_top range across batters:",
      by_batter[("sz_top", "mean")].min().round(2), "-",
      by_batter[("sz_top", "mean")].max().round(2))

TypeError: float() argument must be a string or a real number, not 'NAType'

In [3]:
import pandas as pd
from pathlib import Path
from src.data.ingestion.statcast_client import project_root

raw = project_root() / "data" / "raw"
files = sorted(raw.glob("statcast_*.parquet"))

dtypes = {}
for p in files:
    d = pd.read_parquet(p, columns=["plate_x", "sz_top"]).dtypes
    key = (str(d["plate_x"]), str(d["sz_top"]))
    dtypes.setdefault(key, []).append(p.name)

for k, v in dtypes.items():
    print(k, len(v), "files", "| e.g.", v[0])

('Float64', 'Float64') 182 files | e.g. statcast_2024-03-28_ingested_2026-09-01.parquet
('object', 'object') 4 files | e.g. statcast_2024-07-15_ingested_2026-09-01.parquet


In [4]:
for p in files:
    d = pd.read_parquet(p, columns=["plate_x"]).dtypes
    if str(d["plate_x"]) == "object":
        n = len(pd.read_parquet(p, columns=["plate_x"]))
        print(p.name, n, "rows")

statcast_2024-07-15_ingested_2026-09-01.parquet 0 rows
statcast_2024-07-16_ingested_2026-09-01.parquet 0 rows
statcast_2024-07-17_ingested_2026-09-01.parquet 0 rows
statcast_2024-07-18_ingested_2026-09-01.parquet 0 rows


In [1]:
df = load_all_snapshots()
print(df.shape)
print(df[["plate_x", "plate_z", "sz_top", "sz_bot", "zone"]].dtypes)

NameError: name 'load_all_snapshots' is not defined

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import pandas as pd
from src.utils.pipeline import load_all_snapshots

df = load_all_snapshots()
print(df.shape)

cols = ["plate_x", "plate_z", "sz_top", "sz_bot", "zone", "stand"]
print(df[cols].dtypes)
print()
print(df[cols].isna().mean())
print()
df[cols].describe()

(710632, 119)
plate_x    Float64
plate_z    Float64
sz_top     Float64
sz_bot     Float64
zone         Int64
stand          str
dtype: object

plate_x    0.003753
plate_z    0.003753
sz_top     0.003753
sz_bot     0.003753
zone       0.003753
stand      0.000000
dtype: float64



,plate_x,plate_z,sz_top,sz_bot,zone
count,707965.0,707965.0,707965.0,707965.0,707965.0
mean,0.054566,2.304024,3.408127,1.600586,9.039683
std,0.838486,0.965147,0.191128,0.112178,4.239251
min,-5.380614,-7.65354,2.522457,1.00017,1.0
25%,-0.507343,1.67329,3.28,1.528951,5.0
50%,0.052451,2.31003,3.41,1.6,11.0
75%,0.613492,2.945086,3.53,1.671101,13.0
max,10.203541,12.358563,4.281618,2.135707,14.0


In [2]:
by_batter = (
    df.groupby("batter")[["sz_top", "sz_bot"]]
    .agg(["mean", "std", "count"])
    .round(3)
)
by_batter = by_batter[by_batter[("sz_top", "count")] >= 20]
print(by_batter.head(10).to_string())
print()
print("sz_top range across batters:",
      by_batter[("sz_top", "mean")].min().round(2), "-",
      by_batter[("sz_top", "mean")].max().round(2))

       sz_top              sz_bot             
         mean    std count   mean    std count
batter                                        
444482  3.572  0.068  1014  1.713   0.07  1014
453568  3.507   0.07  1919  1.652  0.065  1919
455117  3.204  0.073   556  1.493  0.061   556
456781  3.212  0.074  1273  1.475  0.048  1273
457705  3.453  0.057  2163  1.659  0.055  2163
457759  3.451  0.088  2320  1.572  0.067  2320
467793  3.414  0.083  2423  1.681  0.065  2423
493329   3.42  0.063   281   1.59  0.054   281
500743  3.277  0.082  1211   1.61  0.074  1211
501303  3.191  0.066   118  1.524  0.052   118

sz_top range across batters: 2.86 - 4.03


In [3]:
print(df["zone"].value_counts().sort_index())

zone
1      29979
2      36180
3      27322
4      41950
5      53207
6      43290
7      33499
8      45192
9      40011
11     76587
12     59779
13     87600
14    133369
Name: count, dtype: Int64


In [4]:
zone_bounds = (
    df.groupby("zone")
    .agg(
        n=("plate_x", "size"),
        x_min=("plate_x", "min"), x_max=("plate_x", "max"),
        z_min=("plate_z", "min"), z_max=("plate_z", "max"),
    )
    .round(2)
)
print(zone_bounds.to_string())

           n  x_min  x_max  z_min  z_max
zone                                    
1      29979  -0.83  -0.28   2.34    4.2
2      36180  -0.28   0.28   2.32   4.17
3      27322   0.28   0.83   2.31   4.19
4      41950  -0.83  -0.28   1.66   3.39
5      53207  -0.28   0.28   1.68   3.33
6      43290   0.28   0.83   1.67   3.43
7      33499  -0.83  -0.28   1.15    2.6
8      45192  -0.28   0.28   1.11   2.58
9      40011   0.28   0.83   1.13   2.61
11     76587  -5.38   -0.0   1.99  12.36
12     59779    0.0   4.25   1.98   8.41
13     87600  -5.11   -0.0  -2.81   3.01
14    133369    0.0   10.2  -7.65   2.99


In [5]:
import numpy as np

# 정의 A: Statcast zone 컬럼
in_zone_statcast = df["zone"].between(1, 9)

# 정의 B: 고정 사각형 (흔히 쓰는 근사)
HALF_PLATE = 0.83   # ft — plate half-width plus ball radius
in_zone_fixed = (
    df["plate_x"].abs() <= HALF_PLATE
) & df["plate_z"].between(1.5, 3.5)

# 정의 C: 타자별 존 (sz_top / sz_bot 사용)
in_zone_batter = (
    df["plate_x"].abs() <= HALF_PLATE
) & (df["plate_z"] >= df["sz_bot"]) & (df["plate_z"] <= df["sz_top"])

for name, mask in [("statcast zone 1-9", in_zone_statcast),
                   ("fixed rectangle", in_zone_fixed),
                   ("batter-specific", in_zone_batter)]:
    print(f"{name:22s} {mask.mean():.1%}  ({mask.sum()})")

print()
print("A vs C disagree on:", (in_zone_statcast != in_zone_batter).sum(), "pitches")
print("B vs C disagree on:", (in_zone_fixed != in_zone_batter).sum(), "pitches")

statcast zone 1-9      49.5%  (350630)
fixed rectangle        49.2%  (348620)
batter-specific        45.5%  (322422)

A vs C disagree on: 28662 pitches
B vs C disagree on: 34990 pitches


In [6]:
taken = df[df["description"].isin(["ball", "called_strike"])].copy()
taken["is_called_strike"] = taken["description"] == "called_strike"

for name, mask in [
    ("statcast zone 1-9", taken["zone"].between(1, 9)),
    ("fixed rectangle", (taken["plate_x"].abs() <= HALF_PLATE) & taken["plate_z"].between(1.5, 3.5)),
    ("batter-specific", (taken["plate_x"].abs() <= HALF_PLATE)
                        & (taken["plate_z"] >= taken["sz_bot"])
                        & (taken["plate_z"] <= taken["sz_top"])),
]:
    agree = (mask == taken["is_called_strike"]).mean()
    print(f"{name:22s} agreement with umpire: {agree:.1%}")


statcast zone 1-9      agreement with umpire: 92.3%
fixed rectangle        agreement with umpire: 92.1%
batter-specific        agreement with umpire: 91.5%


In [7]:
BALL_RADIUS = 0.121   # ft — 2.9 inch diameter / 2

in_zone = (
    (df["plate_x"].abs() <= HALF_PLATE)
    & (df["plate_z"] >= df["sz_bot"] - BALL_RADIUS)
    & (df["plate_z"] <= df["sz_top"] + BALL_RADIUS)
)
print(f"batter-specific + ball radius: {in_zone.mean():.1%}")

batter-specific + ball radius: 49.7%


In [8]:
BALL_RADIUS = 0.121

in_zone_adj = (
    (df["plate_x"].abs() <= HALF_PLATE)
    & (df["plate_z"] >= df["sz_bot"] - BALL_RADIUS)
    & (df["plate_z"] <= df["sz_top"] + BALL_RADIUS)
)
print(f"batter-specific + ball radius: {in_zone_adj.mean():.1%}")
print("vs statcast disagreement:", (in_zone_adj != df["zone"].between(1, 9)).sum())

# 심판 일치율도 다시
taken = df[df["description"].isin(["ball", "called_strike"])]
mask = (
    (taken["plate_x"].abs() <= HALF_PLATE)
    & (taken["plate_z"] >= taken["sz_bot"] - BALL_RADIUS)
    & (taken["plate_z"] <= taken["sz_top"] + BALL_RADIUS)
)
print("umpire agreement:", f"{(mask == (taken['description'] == 'called_strike')).mean():.1%}")

batter-specific + ball radius: 49.7%
vs statcast disagreement: 932
umpire agreement: 92.2%


In [9]:
disagree = df[in_zone_adj != df["zone"].between(1, 9)]
print(len(disagree))
print(disagree[["plate_x", "plate_z", "sz_top", "sz_bot", "zone"]].describe().round(3))

# 경계에서 얼마나 떨어져 있나
dist_x = (disagree["plate_x"].abs() - HALF_PLATE).abs()
dist_z_top = (disagree["plate_z"] - (disagree["sz_top"] + BALL_RADIUS)).abs()
dist_z_bot = (disagree["plate_z"] - (disagree["sz_bot"] - BALL_RADIUS)).abs()
edge_dist = pd.concat([dist_x, dist_z_top, dist_z_bot], axis=1).min(axis=1)
print("distance from nearest boundary (ft):")
print(edge_dist.describe().round(4))

932
       plate_x  plate_z  sz_top  sz_bot    zone
count    932.0    932.0   932.0   932.0   932.0
mean     0.043    2.416   3.412   1.607  12.631
std      0.799    0.888    0.19   0.108   1.142
min      -0.83    1.156   2.775    1.24    11.0
25%     -0.815     1.55   3.281   1.538    12.0
50%      0.742    2.179    3.41   1.608    13.0
75%      0.818    3.352   3.536   1.679    14.0
max       0.83    4.081   4.039   1.912    14.0
distance from nearest boundary (ft):
count     932.0
mean      0.008
std      0.0084
min         0.0
25%      0.0006
50%      0.0049
75%      0.0133
max      0.0342
dtype: Float64
